# 02 — Preprocessing

Preserve zeros, align all product/store histories, and test features available at the forecast origin.

## Imports

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, json, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display


In [2]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'data/calendar.csv').exists())
DATA = ROOT / 'data'
OUT = DATA / 'processed'
RESULTS = ROOT / 'reports/results'
FIGURES = ROOT / 'reports/figures'
for folder in [OUT, RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.venv/matplotlib'))


'/Users/moha/Machine-Learning-Project/Project 1/.venv/matplotlib'

In [3]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':100, 'axes.spines.top':False, 'axes.spines.right':False})
COLORS = ['#2563eb', '#059669', '#ea580c', '#7c3aed']


## Calendar

In [4]:
calendar = pd.read_csv(DATA / 'calendar.csv', parse_dates=['date'])
calendar['day'] = calendar.d.str.removeprefix('d_').astype(int)
calendar = calendar.sort_values('day').reset_index(drop=True)
weeks = pd.Index(sorted(calendar.wm_yr_wk.unique()))
calendar['week_index'] = weeks.get_indexer(calendar.wm_yr_wk)
assert calendar.day.tolist() == list(range(1, len(calendar)+1))
assert calendar.date.diff().dropna().eq(pd.Timedelta(days=1)).all()
display(calendar.head())


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,day,week_index
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,1,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0,2,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0,3,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0,4,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1,5,0


## Product and store identifiers

In [5]:
META = ['id','item_id','dept_id','cat_id','store_id','state_id']
metadata = pd.read_csv(DATA / 'sales_train_validation.csv', usecols=META)
metadata['series_id'] = metadata.id.str.removesuffix('_validation')
series_index = pd.Index(metadata.series_id)
assert series_index.is_unique
assert not metadata.duplicated(['store_id','item_id']).any()
assert not metadata.isna().any().any()


## Validation sales

In [6]:
header = pd.read_csv(DATA / 'sales_train_validation.csv', nrows=0).columns
day_columns = [c for c in header if c.startswith('d_')]
assert day_columns == [f'd_{i}' for i in range(1,len(day_columns)+1)]
values = pd.read_csv(DATA / 'sales_train_validation.csv',
                     usecols=day_columns, dtype='float32').to_numpy()
assert np.isfinite(values).all() and (values >= 0).all()
assert (values == np.floor(values)).all() and values.max() <= 65535
sales = values.astype('uint16')
del values


## Overlapping sales history

In [7]:
evaluation_days = [c for c in pd.read_csv(DATA / 'sales_train_evaluation.csv', nrows=0)
                   if c.startswith('d_')]
full_sales = np.empty((len(metadata), len(evaluation_days)), dtype='uint16')
seen = np.zeros(len(metadata), dtype=bool)


In [8]:
def validate_evaluation_chunk(chunk):
    positions = series_index.get_indexer(chunk.id.str.removesuffix('_evaluation'))
    assert (positions >= 0).all() and len(np.unique(positions)) == len(positions)
    assert not seen[positions].any()
    assert chunk[META[1:]].reset_index(drop=True).equals(
        metadata.iloc[positions][META[1:]].reset_index(drop=True))
    values = chunk[evaluation_days].to_numpy(dtype='float32')
    assert np.isfinite(values).all() and (values >= 0).all()
    assert (values == np.floor(values)).all() and values.max() <= 65535
    np.testing.assert_array_equal(values[:, :len(day_columns)], sales[positions])
    return positions, values.astype('uint16')


In [9]:
reader = pd.read_csv(DATA / 'sales_train_evaluation.csv', chunksize=1000,
                     dtype={d:'float32' for d in evaluation_days})
for chunk in reader:
    positions, values = validate_evaluation_chunk(chunk)
    full_sales[positions] = values
    seen[positions] = True
assert seen.all()
print('All overlapping sales values match after ID alignment.')
del reader, chunk, positions, values


All overlapping sales values match after ID alignment.


## Weekly prices and keys

In [10]:
price_table = pd.read_csv(DATA / 'sell_prices.csv', dtype={
    'store_id':'category', 'item_id':'category',
    'wm_yr_wk':'int32', 'sell_price':'float32'})
assert not price_table.isna().any().any()
assert (price_table.sell_price > 0).all()
assert not price_table.duplicated(['store_id','item_id','wm_yr_wk']).any()
display(price_table.head())


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [11]:
pair_index = pd.MultiIndex.from_frame(metadata[['store_id','item_id']])
price_rows = pair_index.get_indexer(pd.MultiIndex.from_frame(price_table[['store_id','item_id']]))
price_weeks = weeks.get_indexer(price_table.wm_yr_wk)
assert (price_rows >= 0).all() and (price_weeks >= 0).all()
prices = np.full((len(metadata),len(weeks)), np.nan, dtype='float32')
prices[price_rows,price_weeks] = price_table.sell_price.to_numpy()
price_count = len(price_table)
del price_rows, price_weeks, price_table


## Submission format

In [12]:
template = pd.read_csv(DATA / 'sample_submission.csv')
expected_ids = pd.concat([metadata.series_id+'_validation', metadata.series_id+'_evaluation'])
assert template.columns.tolist() == ['id', *[f'F{i}' for i in range(1,29)]]
assert template.id.is_unique and set(template.id) == set(expected_ids)
assert not template.isna().any().any()
print(template.shape)


(60980, 29)


In [13]:
validation_audit = dict(rows=len(metadata), days=sales.shape[1],
    zeros=int((sales==0).sum()), units=int(sales.sum(dtype='int64')))
audit = dict(sales={'validation':validation_audit,
    'evaluation':dict(rows=len(metadata),days=full_sales.shape[1])},
    prices=dict(rows=price_count), submission=dict(rows=len(template)))
data = SimpleNamespace(root=ROOT, sales=sales, prices=prices,
                       metadata=metadata, calendar=calendar, audit=audit)


## Encode known metadata and calendar categories

In [14]:
for column in ['item_id','dept_id','cat_id','store_id','state_id']:
    metadata[column+'_code'] = pd.Categorical(metadata[column],
        categories=sorted(metadata[column].unique())).codes
for column in ['event_name_1','event_name_2']:
    calendar[column+'_code'] = pd.Categorical(calendar[column].fillna('No event')).codes


## Save compact arrays

In [15]:
np.save(OUT / 'sales.npy',full_sales)
np.save(OUT / 'prices.npy',prices)
metadata.to_csv(OUT / 'metadata.csv',index=False)
calendar.to_csv(OUT / 'calendar.csv',index=False)
sales = np.load(OUT / 'sales.npy',mmap_mode='r')
del full_sales
display(pd.DataFrame({'array':['sales','prices'],
    'shape':[str(sales.shape),str(prices.shape)],
    'MB':[sales.nbytes/1024**2,prices.nbytes/1024**2]}))


,array,shape,MB
0,sales,"(30490, 1941)",112.878971
1,prices,"(30490, 282)",32.799454


## Comparison settings

In [16]:
config = json.loads((ROOT / 'configs/comparison.json').read_text())
display(pd.Series(config).to_frame('setting'))


,setting
horizon,28
development_cutoffs,"[1857, 1885]"
test_cutoff,1913
history_days,728
training_stride,13
seed,42
n_estimators,200
num_leaves,31
learning_rate,0.05
baseline_floor,0.05


## Calendar features

In [17]:
CATEGORICAL = ['item_id_code','store_id_code','dept_id_code','cat_id_code',
               'state_id_code','weekday','month','event_1','event_2']


In [18]:
def calendar_features(rows, days):
    cal = calendar.iloc[days-1]
    meta = metadata.iloc[rows]
    f = meta[CATEGORICAL[:5]].reset_index(drop=True).astype('int16')
    f['weekday'] = cal.wday.to_numpy(dtype='int8')
    f['month'] = cal.month.to_numpy(dtype='int8')
    f['event_1'] = cal.event_name_1_code.to_numpy(dtype='int16')
    f['event_2'] = cal.event_name_2_code.to_numpy(dtype='int16')
    f['trend'] = (days-1).astype('float32')
    angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25
    f['annual_sin'], f['annual_cos'] = np.sin(angle).astype('float32'), np.cos(angle).astype('float32')
    f['snap'] = np.select([meta.state_id.to_numpy()==s for s in ['CA','TX','WI']],
                          [cal['snap_'+s].to_numpy() for s in ['CA','TX','WI']]).astype('int8')
    return f


## Lag and rolling features

In [19]:
def demand_features(rows, days):
    t = days-1
    f = {f'lag_{lag}':sales[rows,t-lag].astype('float32') for lag in [28,35,42,56]}
    window = sales[rows[:,None],t[:,None]-28-np.arange(28)]
    f['mean_7'] = window[:,:7].mean(axis=1,dtype='float32')
    f['mean_28'] = window.mean(axis=1,dtype='float32')
    f['std_28'] = window.std(axis=1,dtype='float32')
    f['zero_rate_28'] = (window==0).mean(axis=1).astype('float32')
    return pd.DataFrame(f)


## Features available at the forecast origin

In [20]:
def origin_features(rows, days, origins):
    rows,days,origins = map(np.asarray,(rows,days,origins))
    assert ((days-origins>=1)&(days-origins<=28)).all() and origins.min()>=56
    f = pd.concat([calendar_features(rows,days),demand_features(rows,days)],axis=1)
    recent = sales[rows[:,None],origins[:,None]-1-np.arange(28)]
    last_weekday = origins-1-((origins-days)%7)
    weekly = sales[rows[:,None],last_weekday[:,None]-7*np.arange(8)]
    f['horizon'] = (days-origins).astype('int8')
    f['origin_mean_7'] = recent[:,:7].mean(axis=1,dtype='float32')
    f['origin_mean_28'] = recent.mean(axis=1,dtype='float32')
    f['origin_zero_rate_28'] = (recent==0).mean(axis=1).astype('float32')
    f['weekday_baseline'] = weekly.mean(axis=1,dtype='float32')
    return f


## Training dates and forecast indices

In [21]:
def training_indices(cutoff):
    rows,days = [],[]
    start = cutoff-config['history_days']+1
    for offset in range(config['training_stride']):
        group = np.arange(offset,len(metadata),config['training_stride'])
        targets = np.arange(start+offset,cutoff+1,config['training_stride'])
        rows.append(np.repeat(group,len(targets)))
        days.append(np.tile(targets,len(group)))
    return np.concatenate(rows),np.concatenate(days)


In [22]:
def forecast_indices(sample_rows, cutoff):
    rows = np.repeat(np.asarray(sample_rows),config['horizon'])
    days = np.tile(np.arange(cutoff+1,cutoff+config['horizon']+1),len(sample_rows))
    return rows,days,np.full(len(rows),cutoff)


## Feature example

In [23]:
rows,days,origins = forecast_indices(np.arange(12),config['development_cutoffs'][0])
features = origin_features(rows,days,origins)
display(features.head())
display(features.dtypes.to_frame('dtype'))
features.to_csv(OUT / 'feature_sample.csv',index=False)


/var/folders/lq/jhztqld97t305sjbgrvcllph0000gp/T/ipykernel_21497/2701393547.py:10: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25


,item_id_code,store_id_code,dept_id_code,cat_id_code,state_id_code,weekday,month,event_1,event_2,trend,...,lag_56,mean_7,mean_28,std_28,zero_rate_28,horizon,origin_mean_7,origin_mean_28,origin_zero_rate_28,weekday_baseline
0,1437,0,3,1,0,3,2,19,3,1857.0,...,2.0,0.714286,0.785714,1.205853,0.535714,1,1.714286,1.035714,0.428571,1.125
1,1437,0,3,1,0,4,3,19,3,1858.0,...,0.0,0.714286,0.821429,1.196828,0.500000,2,1.714286,1.035714,0.428571,1.375
2,1437,0,3,1,0,5,3,19,3,1859.0,...,1.0,0.714286,0.821429,1.196828,0.500000,3,1.714286,1.035714,0.428571,0.625
3,1437,0,3,1,0,6,3,19,3,1860.0,...,0.0,0.714286,0.857143,1.186661,0.464286,4,1.714286,1.035714,0.428571,0.625
4,1437,0,3,1,0,7,3,19,3,1861.0,...,1.0,0.714286,0.821429,1.196828,0.500000,5,1.714286,1.035714,0.428571,0.625


,dtype
item_id_code,int16
store_id_code,int16
dept_id_code,int16
cat_id_code,int16
state_id_code,int16
weekday,int8
month,int8
event_1,int16
event_2,int16
trend,float32


## Future-sales leakage check

In [24]:
before = origin_features(rows,days,origins)
original_sales = sales
try:
    sales = np.array(original_sales[:12],copy=True)
    sales[:,config['development_cutoffs'][0]:] = 65535
    after = origin_features(rows,days,origins)
    pd.testing.assert_frame_equal(before,after)
finally:
    sales = original_sales
print('Future sales do not change forecast features.')


Future sales do not change forecast features.


/var/folders/lq/jhztqld97t305sjbgrvcllph0000gp/T/ipykernel_21497/2701393547.py:10: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25
/var/folders/lq/jhztqld97t305sjbgrvcllph0000gp/T/ipykernel_21497/2701393547.py:10: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  angle = 2*np.pi*cal.date.dt.dayofyear.to_numpy()/365.25


## Include every product/store series

In [25]:
cutoff = min(config['development_cutoffs'])
panel = metadata[['series_id','store_id','cat_id']].copy()
panel['mean_training_sales'] = sales[:,cutoff-config['history_days']:cutoff].mean(axis=1)
panel = panel.rename_axis('row').reset_index()
assert len(panel)==len(metadata)==30490
assert panel.row.tolist()==list(range(len(metadata)))
panel.to_csv(OUT / 'comparison_series.csv',index=False)
display(panel.groupby(['store_id','cat_id']).size().unstack())


cat_id,FOODS,HOBBIES,HOUSEHOLD
store_id,,,
CA_1,1437,565,1047
CA_2,1437,565,1047
CA_3,1437,565,1047
CA_4,1437,565,1047
TX_1,1437,565,1047
TX_2,1437,565,1047
TX_3,1437,565,1047
WI_1,1437,565,1047
WI_2,1437,565,1047


## Time splits

In [26]:
splits=[]
for phase,cutoffs in [('development',config['development_cutoffs']),('local_test',[config['test_cutoff']])]:
    for cutoff in cutoffs:
        splits.append({'phase':phase,'cutoff':cutoff,
            'forecast_start':calendar.date.iloc[cutoff],
            'forecast_end':calendar.date.iloc[cutoff+27]})
splits=pd.DataFrame(splits)
splits.to_csv(OUT / 'comparison_splits.csv',index=False)
display(splits)


,phase,cutoff,forecast_start,forecast_end
0,development,1857,2016-02-29,2016-03-27
1,development,1885,2016-03-28,2016-04-24
2,local_test,1913,2016-04-25,2016-05-22
